# Phase 2-2: Gaussian Noise Attack

Generates noisy versions of the dataset at different sigma levels.
Includes quality metrics (PSNR, SSIM) and visualizations.

In [ ]:
import os
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
from PIL import Image
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# mount drive
from google.colab import drive
drive.mount('/content/drive')

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("Imports OK")
print(f"CWD: {os.getcwd()}")

In [ ]:
CONFIG = {
    # source (clean data)
    "sourcePath": "/content/drive/MyDrive/Colab Notebooks/data/training",
    
    # output
    "outputPath": "/content/drive/MyDrive/Colab Notebooks/data/adversarial/noise",
    
    # noise levels (sigma as fraction of 255)
    "sigmaLevels": [0.25],
    "seeds": [42],
    
    # splits to process
    "splits": ["train", "val", "test"],
    
    # visualization
    "numSamples": 5,
}

print("Config:")
print(f"  Sigma levels: {CONFIG['sigmaLevels']}")
print(f"  Seeds: {CONFIG['seeds']}")
print(f"  Splits: {CONFIG['splits']}")

In [ ]:
def createOutputDirs(basePath, sigma, seeds, splits):
    """Create directory structure for a sigma level.
    
    Args:
        basePath: root output directory
        sigma: noise level
        seeds: list of random seeds
        splits: list of splits (train/val/test)
    
    Returns:
        list of created paths
    """
    created = []
    
    for seedIdx, seed in enumerate(seeds):
        for split in splits:
            imgDir = Path(basePath) / f"sigma_{sigma:.3f}" / f"seed_{seedIdx}" / split / "images"
            lblDir = Path(basePath) / f"sigma_{sigma:.3f}" / f"seed_{seedIdx}" / split / "labels"
            
            imgDir.mkdir(parents=True, exist_ok=True)
            lblDir.mkdir(parents=True, exist_ok=True)
            
            created.append(imgDir)
            created.append(lblDir)
    
    return created


def createVizDir(basePath):
    """Create visualization directory."""
    vizPath = Path(basePath) / "visualizations"
    vizPath.mkdir(parents=True, exist_ok=True)
    
    (vizPath / "grids").mkdir(exist_ok=True)
    (vizPath / "metrics").mkdir(exist_ok=True)
    
    return vizPath

In [ ]:
def addGaussianNoise(imgArray, sigma, seed=None):
    """Add Gaussian noise to an image.
    
    Args:
        imgArray: numpy array in [0, 255]
        sigma: noise std (normalized, so 0.1 means 0.1*255 = 25.5 pixel std)
        seed: random seed for reproducibility
    
    Returns:
        noisy image as uint8 array
    """
    if seed is not None:
        np.random.seed(seed)
    
    # convert sigma to pixel space
    sigmaPixels = sigma * 255
    
    # generate and add noise
    noise = np.random.normal(0, sigmaPixels, imgArray.shape)
    noisy = imgArray + noise
    
    # clip to valid range
    noisy = np.clip(noisy, 0, 255)
    
    return noisy.astype(np.uint8)


def saveImage(imgArray, path):
    """Save numpy array as image."""
    try:
        Image.fromarray(imgArray).save(path)
        return True
    except Exception as e:
        print(f"Error saving: {e}")
        return False

In [ ]:
def getLabelPath(imgPath):
    """Get corresponding label path for an image."""
    imgPath = Path(imgPath)
    lblName = imgPath.stem + ".txt"
    lblPath = imgPath.parent.parent / "labels" / lblName
    return lblPath


def copyLabel(srcImgPath, dstLblDir):
    """Copy label file to destination."""
    lblSrc = getLabelPath(srcImgPath)
    
    if lblSrc.exists():
        lblDst = Path(dstLblDir) / lblSrc.name
        try:
            shutil.copy2(lblSrc, lblDst)
            return True
        except Exception as e:
            print(f"Error copying label: {e}")
            return False
    
    return True  # no label is ok

In [ ]:
def processImage(imgPath, outDir, sigma, seed):
    """Process one image with Gaussian noise.
    
    Args:
        imgPath: source image path
        outDir: output directory
        sigma: noise level
        seed: random seed
    
    Returns:
        dict with processing result
    """
    img = Image.open(imgPath)
    imgArray = np.array(img)
    
    noisyImg = addGaussianNoise(imgArray, sigma, seed)
    
    outPath = outDir / "images" / imgPath.name
    imgSaved = saveImage(noisyImg, outPath)
    lblCopied = copyLabel(imgPath, outDir / "labels")
    
    return {
        'filename': imgPath.name,
        'sigma': sigma,
        'seed': seed,
        'imageSaved': imgSaved,
        'labelCopied': lblCopied
    }


def processBatch(imgPaths, outBase, sigma, seed, split):
    """Process a batch of images."""
    results = []
    
    for imgPath in tqdm(imgPaths, desc=f"σ={sigma:.3f}, {split}"):
        result = processImage(imgPath, outBase, sigma, seed)
        result['split'] = split
        results.append(result)
    
    return results

In [ ]:
def computePsnr(original, noisy):
    """Compute Peak Signal-to-Noise Ratio."""
    return psnr(original, noisy)


def computeSsim(original, noisy):
    """Compute Structural Similarity Index."""
    if len(original.shape) == 3:
        origGray = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)
        noisyGray = cv2.cvtColor(noisy, cv2.COLOR_RGB2GRAY)
    else:
        origGray = original
        noisyGray = noisy
    
    return ssim(origGray, noisyGray)


def computeMetrics(origPath, noisyPath):
    """Compute quality metrics between original and noisy images."""
    original = np.array(Image.open(origPath))
    noisy = np.array(Image.open(noisyPath))
    
    return {
        'psnr': computePsnr(original, noisy),
        'ssim': computeSsim(original, noisy)
    }

In [ ]:
def plotNoiseProgression(imgPath, sigmaLevels, seeds, savePath=None):
    """Create a grid showing noise at different levels.
    
    Args:
        imgPath: path to original image
        sigmaLevels: list of sigma values
        seeds: list of seeds
        savePath: where to save figure
    
    Returns:
        matplotlib figure
    """
    original = np.array(Image.open(imgPath))
    
    nRows = len(seeds)
    nCols = len(sigmaLevels) + 1
    
    fig, axes = plt.subplots(nRows, nCols, figsize=(3 * nCols, 3 * nRows))
    
    if nRows == 1:
        axes = axes.reshape(1, -1)
    
    # first column: original
    for i in range(nRows):
        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f'Original\n(Seed {seeds[i]})')
        axes[i, 0].axis('off')
    
    # add noisy versions
    for i, seed in enumerate(seeds):
        for j, sigma in enumerate(sigmaLevels):
            noisy = addGaussianNoise(original, sigma, seed)
            
            psnrVal = computePsnr(original, noisy)
            ssimVal = computeSsim(original, noisy)
            
            axes[i, j+1].imshow(noisy)
            axes[i, j+1].set_title(f'σ={sigma}')
            axes[i, j+1].axis('off')
            
            axes[i, j+1].text(
                0.5, -0.15, f'PSNR: {psnrVal:.1f}\nSSIM: {ssimVal:.3f}',
                transform=axes[i, j+1].transAxes, ha='center', fontsize=8
            )
    
    plt.suptitle(f'Noise Progression: {Path(imgPath).name}', fontsize=12)
    plt.tight_layout()
    
    if savePath:
        fig.savefig(savePath, dpi=150, bbox_inches='tight')
    
    return fig

In [ ]:
def getSplitImages(sourcePath, split):
    """Get all images from a dataset split.
    
    Args:
        sourcePath: root dataset path
        split: split name (train/val/test)
    
    Returns:
        sorted list of image paths
    """
    srcDir = Path(sourcePath) / split / "images"
    
    images = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        images.extend(srcDir.glob(ext))
    
    return sorted(images)


def countFiles(path, split):
    """Count images and labels in a split."""
    path = Path(path) / split
    
    nImgs = len(list((path / "images").glob("*")))
    nLbls = len(list((path / "labels").glob("*.txt")))
    
    return {'images': nImgs, 'labels': nLbls}

In [ ]:
def processSigmaLevel(srcImages, outBase, sigma, seeds, split):
    """Process all images for one noise level.
    
    Args:
        srcImages: list of source image paths
        outBase: base output directory
        sigma: noise level
        seeds: list of seeds
        split: split name
    
    Returns:
        combined results from all seeds
    """
    allResults = []
    
    for seedIdx, seed in enumerate(seeds):
        outDir = outBase / f"sigma_{sigma:.3f}" / f"seed_{seedIdx}" / split
        results = processBatch(srcImages, outDir, sigma, seed, split)
        allResults.extend(results)
    
    return allResults


def processSplit(sourcePath, outputPath, split, sigmaLevels, seeds):
    """Process entire dataset split with all noise levels.
    
    Args:
        sourcePath: source dataset path
        outputPath: output directory
        split: split name
        sigmaLevels: list of sigma values
        seeds: list of seeds
    
    Returns:
        results organized by sigma
    """
    print(f"\nProcessing {split}...")
    
    images = getSplitImages(sourcePath, split)
    print(f"Found {len(images)} images")
    
    splitResults = {}
    
    for sigma in sigmaLevels:
        print(f"\nσ={sigma:.3f}...")
        results = processSigmaLevel(
            images, Path(outputPath), sigma, seeds, split
        )
        splitResults[f"sigma_{sigma:.3f}"] = results
    
    return splitResults

In [ ]:
def sampleMetrics(sourcePath, outputPath, split, sigma, seedIdx, nSamples=10):
    """Sample quality metrics for a noise configuration.
    
    Args:
        sourcePath: original dataset path
        outputPath: noisy dataset path
        split: split name
        sigma: noise level
        seedIdx: seed index
        nSamples: number of samples
    
    Returns:
        list of metric dicts
    """
    srcImages = getSplitImages(sourcePath, split)
    indices = np.random.choice(
        len(srcImages), min(nSamples, len(srcImages)), replace=False
    )
    
    metrics = []
    for idx in indices:
        original = srcImages[idx]
        noisyPath = (
            Path(outputPath) / f"sigma_{sigma:.3f}" /
            f"seed_{seedIdx}" / split / "images" / original.name
        )
        
        if noisyPath.exists():
            m = computeMetrics(original, noisyPath)
            m['filename'] = original.name
            m['sigma'] = sigma
            m['seed'] = seedIdx
            m['split'] = split
            metrics.append(m)
    
    return metrics


def aggregateMetrics(metricsList):
    """Aggregate metrics by sigma level."""
    df = pd.DataFrame(metricsList)
    
    summary = df.groupby('sigma').agg({
        'psnr': ['mean', 'std', 'min', 'max'],
        'ssim': ['mean', 'std', 'min', 'max']
    }).round(3)
    
    return summary

In [ ]:
def main():
    """Run the full noise generation pipeline."""
    print("\n" + "="*60)
    print("GAUSSIAN NOISE GENERATION")
    print("="*60)
    
    # create directories
    print("\n[Step 1] Creating directories...")
    for sigma in CONFIG["sigmaLevels"]:
        createOutputDirs(
            CONFIG["outputPath"], sigma,
            CONFIG["seeds"], CONFIG["splits"]
        )
    vizPath = createVizDir(CONFIG["outputPath"])
    print(f"Output: {CONFIG['outputPath']}")
    
    # process each split
    allResults = {}
    allMetrics = []
    
    for split in CONFIG["splits"]:
        splitResults = processSplit(
            CONFIG["sourcePath"], CONFIG["outputPath"],
            split, CONFIG["sigmaLevels"], CONFIG["seeds"]
        )
        allResults[split] = splitResults
        
        # sample metrics
        for sigma in CONFIG["sigmaLevels"]:
            for seedIdx in range(len(CONFIG["seeds"])):
                metrics = sampleMetrics(
                    CONFIG["sourcePath"], CONFIG["outputPath"],
                    split, sigma, seedIdx, nSamples=10
                )
                allMetrics.extend(metrics)
    
    # save metrics
    print("\n[Step 3] Saving metrics...")
    metricsDf = pd.DataFrame(allMetrics)
    metricsDf.to_csv(vizPath / "metrics" / "quality_metrics.csv", index=False)
    
    # print summary
    print("\n" + "="*50)
    print("QUALITY METRICS")
    print("="*50)
    summary = aggregateMetrics(allMetrics)
    print(summary)
    
    print("\nDone!")
    print(f"Output: {CONFIG['outputPath']}")
    
    return {'results': allResults, 'metrics': metricsDf, 'summary': summary}


# run it
pipelineResults = main()

In [ ]:
def validateDataset(outputPath, sigma, seedIdx, split):
    """Validate a single noise variant."""
    base = Path(outputPath) / f"sigma_{sigma:.3f}" / f"seed_{seedIdx}" / split
    
    nImgs = len(list((base / "images").glob("*")))
    nLbls = len(list((base / "labels").glob("*.txt")))
    
    return {
        'sigma': sigma,
        'seed': seedIdx,
        'split': split,
        'images': nImgs,
        'labels': nLbls,
        'valid': nImgs == nLbls
    }


def validateAll():
    """Validate all generated datasets."""
    validations = []
    
    for sigma in CONFIG["sigmaLevels"]:
        for seedIdx in range(len(CONFIG["seeds"])):
            for split in CONFIG["splits"]:
                val = validateDataset(
                    CONFIG["outputPath"], sigma, seedIdx, split
                )
                validations.append(val)
    
    valDf = pd.DataFrame(validations)
    
    print("\n" + "="*50)
    print("VALIDATION")
    print("="*50)
    print(valDf.groupby(['sigma', 'split'])['images'].sum())
    
    if valDf['valid'].all():
        print("\nAll datasets valid!")
    else:
        print("\nSome datasets have mismatched counts!")
    
    return valDf


validationResults = validateAll()

In [ ]:
def generateVisualizations():
    """Generate sample visualizations."""
    print("\n" + "="*50)
    print("VISUALIZATIONS")
    print("="*50)
    
    vizBase = Path(CONFIG["outputPath"]) / "visualizations"
    vizPaths = []
    
    # get sample images from val set
    valImages = getSplitImages(CONFIG["sourcePath"], "val")
    samples = valImages[:min(3, len(valImages))]
    
    for imgPath in samples:
        savePath = vizBase / "grids" / f"progression_{imgPath.stem}.png"
        
        fig = plotNoiseProgression(
            imgPath,
            CONFIG["sigmaLevels"],
            CONFIG["seeds"][:2],
            savePath
        )
        plt.show()
        plt.close()
        
        vizPaths.append(savePath)
        print(f"Saved: {savePath.name}")
    
    return vizPaths


vizFiles = generateVisualizations()